# XYZ2 MDR Plotting Notebook

Reload saved simulation data and generate threshold plots.

## Run simulations first (terminal)

Run both simulation sets before using reload/plot cells:

```bash
python scripts/run_sweeps_no_spam.py
python scripts/run_sweeps_with_spam.py
```

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path('.').resolve()
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from xyz2_mdr.mdr_noise_sweep import MdrNoiseSweep


In [2]:
# User config
distances = [3, 5, 7, 9, 11]
results_dir = Path('data/simulation_results')
plots_dir = Path('data/plots')
plots_dir.mkdir(parents=True, exist_ok=True)

# Set to None to load latest per (noise_model, distance).
# Set to 0.0 or 1.339e-3 to select a specific SPAM run.
p_spam_filter = None


In [3]:
def _close(a: float, b: float, tol: float = 1e-15) -> bool:
    return abs(float(a) - float(b)) <= tol


def _resolve_result_csv(
    noise_model: str,
    distance: int,
    p_spam: float | None = None,
) -> Path:
    legacy = results_dir / f'results_{noise_model}_d{distance}.csv'
    if legacy.exists() and p_spam is None:
        return legacy

    spec_files = sorted(
        results_dir.glob(f'results_{noise_model}_d{distance}_*.spec.json')
    )

    matches = []
    for spec_path in spec_files:
        spec = json.loads(spec_path.read_text(encoding='utf-8'))
        if p_spam is not None:
            val = float(spec.get('p_spam', -1.0))
            if not _close(val, p_spam):
                continue
        csv_path = spec_path.with_suffix('').with_suffix('.csv')
        if csv_path.exists():
            matches.append(csv_path)

    if not matches:
        msg = f'No saved results for {noise_model}, d={distance}'
        if p_spam is not None:
            msg += f', p_spam={p_spam:g}'
        raise FileNotFoundError(msg)

    matches.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return matches[0]


In [4]:
# ==============================================================================
# 4. Reload Data & Plot Results
# ==============================================================================
# We reconstruct MdrNoiseSweep objects from saved CSVs.

sweeps_z_type_loaded = {}
sweeps_pure_z_loaded = {}
sweeps_unbiased_loaded = {}

print('Reloading data from CSVs...')
for d in distances:
    # 1. Z Type Noise
    try:
        csv_path = _resolve_result_csv('z_type', d, p_spam_filter)
        sweeps_z_type_loaded[f'Z Type (d={d})'] = MdrNoiseSweep(
            load_data_filename=csv_path
        )
    except FileNotFoundError:
        print(f'Warning: Z Type data for d={d} not found.')

    # 2. Pure Z Noise
    try:
        csv_path = _resolve_result_csv('pure_z', d, p_spam_filter)
        sweeps_pure_z_loaded[f'Pure Z (d={d})'] = MdrNoiseSweep(
            load_data_filename=csv_path
        )
    except FileNotFoundError:
        print(f'Warning: Pure Z data for d={d} not found.')

    # 3. Unbiased Noise
    try:
        csv_path = _resolve_result_csv('unbiased', d, p_spam_filter)
        sweeps_unbiased_loaded[f'Unbiased (d={d})'] = MdrNoiseSweep(
            load_data_filename=csv_path
        )
    except FileNotFoundError:
        print(f'Warning: Unbiased data for d={d} not found.')

print('Data reload complete. Generating plots...')


Reloading data from CSVs...
Data reload complete. Generating plots...


In [5]:
# ==============================================================================
# 5. Generate Plots
# ==============================================================================

suffix = ''
if p_spam_filter is not None:
    suffix = f'_pspam{p_spam_filter:.3e}'.replace('+', '')

# 1. Plot Z Type Noise Threshold
MdrNoiseSweep.plot_error_multi(
    sweeps_z_type_loaded,
    category='logical',
    rounds=[1],
    subset=['Logical X'],
    overlay=False,
    log_x=True,
    save_path=plots_dir / f'threshold_z_type_noise{suffix}.pdf',
)
print(f'Saved threshold_z_type_noise{suffix}.pdf')

# 2. Plot Pure Z Noise Threshold
MdrNoiseSweep.plot_error_multi(
    sweeps_pure_z_loaded,
    category='logical',
    rounds=[1],
    subset=['Logical X'],
    overlay=False,
    log_x=True,
    save_path=plots_dir / f'threshold_pure_z_noise{suffix}.pdf',
)
print(f'Saved threshold_pure_z_noise{suffix}.pdf')

# 3. Plot Unbiased Noise Threshold
MdrNoiseSweep.plot_error_multi(
    sweeps_unbiased_loaded,
    category='logical',
    rounds=[1],
    subset=['Logical X'],
    overlay=False,
    log_x=True,
    save_path=plots_dir / f'threshold_unbiased_noise{suffix}.pdf',
)
print(f'Saved threshold_unbiased_noise{suffix}.pdf')


ValueError: No sweeps provided